# Best-of-N speed benchmark: across models

Time `bon_search_v1.best_of_n_v1` on the same set of MATH problems
across several causal LMs. Each model is loaded under vLLM, warmed
up, timed over `num_trials` runs, then unloaded before the next.

Pair with `benchmark_bon_quant_speed_v1.ipynb` for the
quantization-axis sweep.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import sys
sys.path.append("..")

import gc
import statistics
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1
from utils.load_data import load_data_hf

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

# Models to benchmark — same prompts, same config; only this varies
llm_dirs = [
    os.path.join(base_dir, "Llama3.2-1B-Instruct"),
    os.path.join(base_dir, "Llama3.2-3B-Instruct"),
    os.path.join(base_dir, "Qwen2.5-3B-Instruct"),
    os.path.join(base_dir, "Qwen2.5-7B-Instruct"),
]

In [ ]:
# Best-of-N search params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs (all in one place so the matrix is easy to vary)
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 10                 # cap on questions per benchmark
num_trials = 2                     # timed runs per model
warmup = 1                         # untimed warmup runs per model
llm_gpu_memory_utilization = 0.7

In [ ]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

## Helpers

In [ ]:
def gpu_mem_used_gb(device=0):
    """Driver-level used GPU memory; sees both PyTorch and vLLM allocs."""
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


def benchmark_model(llm_dir, config, prompts, num_trials, warmup=1):
    """Load `llm_dir` under vLLM, warm up, time `num_trials` runs of
    best_of_n_v1, then tear down. Returns (model_name, trial_times).
    """
    model_name = os.path.basename(llm_dir.rstrip('/'))
    print(f"\n=== {model_name} ===")

    # enforce_eager=True disables CUDA graphs - skips cudagraph capture cost
    # on every model load, giving more stable latency at small num_trials.
    llm = LLM(
        model=llm_dir,
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype="float16",
        seed=config.seed,
    )
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory used: {gpu_mem_used_gb():.2f} GB")

    # Warmup (untimed) - absorbs first-call init inside best_of_n_v1
    for w in range(warmup):
        bon_search_v1.best_of_n_v1(prompts, config, llm, 10_000 + w)

    times = []
    for trial_idx in range(num_trials):
        start = time.perf_counter()
        bon_search_v1.best_of_n_v1(prompts, config, llm, trial_idx)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        print(
            f"  trial {trial_idx}: {elapsed:>7.2f}s total, "
            f"{elapsed / len(prompts):.4f}s/question"
        )

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    return model_name, times

## Run benchmark

One model at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [ ]:
results = []
for llm_dir in llm_dirs:
    name, times = benchmark_model(
        llm_dir, config, batch_of_questions, num_trials, warmup=warmup,
    )
    results.append((name, times))

## Summary

In [ ]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'model':<24}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<24}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )